In [7]:
import os
import sys
sys.path.append(os.path.abspath('..'))
import logging
import warnings
import re
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from src.build_dataset import get_file_pairs, merge_qa_data, detect_exercise_type, find_answer_index, apply_reference_tag

# --- Setup Warnings ---
warnings.filterwarnings('ignore', category=SyntaxWarning, message='invalid escape sequence')

# --- Setup Logger ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# --- Load Environment Variables ---
load_dotenv()

True

In [8]:
data_path = os.getenv("DATA_DIR")

if data_path:
    data_dir = Path(data_path)
    logger.info(f"Ξεκινάει η αναζήτηση στον φάκελο: {data_dir}")
    
    all_pairs = get_file_pairs (data_dir, target_school="GEL")
    logger.info(f"Βρέθηκαν συνολικά {len(all_pairs)} ζευγάρια αρχείων (JSON/MD).")
else:
    logger.error("Το DATA_DIR δεν βρέθηκε στο .env αρχείο!")

2026-04-03 08:53:15 - INFO - Ξεκινάει η αναζήτηση στον φάκελο: C:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πανελλήνιες\panellinies_exams_dataset\data
2026-04-03 08:53:15 - INFO - Βρέθηκαν συνολικά 59 ζευγάρια αρχείων (JSON/MD).


In [9]:
main_dataset = []

for pair in all_pairs:
    json_path = pair["json"]
    md_path = pair["md"]
    
    qa_list = merge_qa_data(json_path, md_path)
    
    main_dataset.extend(qa_list)

logger.info (f"Η ενοποίηση ολοκληρώθηκε! Βρέθηκαν συνολικά {len(main_dataset)} ερωτήσεις-απαντήσεις!")

2026-04-03 08:53:17 - INFO - Η ενοποίηση ολοκληρώθηκε! Βρέθηκαν συνολικά 1318 ερωτήσεις-απαντήσεις!


In [10]:
for item in main_dataset:
    q_text = item.get("question","")
    q_choices = item.get("choices",[])
    ans_text = item.get("answer","")
    images_list = item.get("images", [])
    marks = item.get("mark", [])
    
    form_type = detect_exercise_type(q_text,q_choices)
    item["format"] = form_type
    ans_idx = find_answer_index(q_choices,ans_text)
    item["answer_index"] = ans_idx
    item["reference"] = apply_reference_tag(item)
    
    #parsing image description and transcription
    all_descriptions = []
    all_transcriptions = []
    all_paths = []
    
    for img_dict in images_list:
        desc = img_dict.get("description","")
        if desc:
            all_descriptions.append(desc)
        transc = img_dict.get("transcription",[])
        if transc and isinstance(transc, list):
            joined_transc = ", ".join(transc)
            all_transcriptions.append(joined_transc)
        
        img_path = img_dict.get("path", "")
        if img_path:
            all_paths.append(img_path)
    
    mark_list = []
    
    for mark_text in marks:
        match = re.search(r'\d+\.?\d*', str(mark_text))
        if match:
            num_str = match.group()
            if "." in num_str:
                mark_list.append(float(num_str))
            else:
                mark_list.append(int(num_str))
    
    if len(mark_list) == 1:
        item["points"] = mark_list[0]
    elif len(mark_list) > 1:
        item["points"] = sum(mark_list)
    else:
        item["points"] = None
            
    item["image_description"] = " | ".join(all_descriptions)
    item["image_transcription"] = " | ".join(all_transcriptions)
    item["images"] = all_paths
    item.pop("mark", None)

In [ ]:
images_found = 0
print("--- Ερωτήσεις που βρέθηκαν να έχουν εικόνες ---")

for item in main_dataset:
    imgs = item.get("images", [])
    
    if isinstance(imgs, list) and len(imgs) > 0:
        images_found += 1
        print(f"ID: {item.get('id')} στο μάθημα {item.get('subject')} ({item.get('year')}) - Περιέχει {len(imgs)} εικόνα/ες")

print(f"\nΣυνολικά βρέθηκαν {images_found} ερωτήσεις (IDs) με εικόνες.")

In [12]:
results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

output_file = results_dir / "test_dataset.xlsx"

df = pd.DataFrame(main_dataset)
df.to_excel(output_file, index=False)

logger.info (f"Tο αρχείο δημιουργήθηκε επιτυχώς στο: {output_file.resolve()}!")

df.head()

2026-04-03 08:53:33 - INFO - Tο αρχείο δημιουργήθηκε επιτυχώς στο: C:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πανελλήνιες\panellinies_exams_dataset\results\test_dataset.xlsx!


,id,question,input,choices,images,answer,subject,year,school_type,format,answer_index,reference,points,image_description,image_transcription
0,Α1.α.1,Ποια είναι η κύρια αιτία η οποία εμποδίζει του...,Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,"[α. Οι αλυσίδες στους αυχένες τους., β. Το σκο...",[],α,arxaia,2025,GEL,multiple_choice,0.0,passage,2.0,,
1,Α1.α.2,Πού βρίσκεται το πυρ σε σχέση με τους δεσμώτες;,Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,"[α. Μπροστά τους, χαμηλά., β. Επάνω και πίσω τ...",[],β,arxaia,2025,GEL,multiple_choice,1.0,passage,2.0,,
2,Α1.α.3,"Τα «σκεύη», οι «ἀνδριάντες» και τα «ἄλλα ζῷα» ...",Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,"[α. είναι πραγματικά ζώα της σπηλιάς., β. δημι...",[],β,arxaia,2025,GEL,multiple_choice,1.0,passage,2.0,,
3,Α1.β,"«παρ’ ἣν», «ὑπὲρ ὧν»: Σε ποια λέξη του αρχαίου...",Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,[],[],παρ ́ἥν: αναφέρεται στην ὁδόν\nὑπέρ ὧν: αναφέρ...,arxaia,2025,GEL,open_ended,NaN,passage,4.0,,
4,B1,"Ποιος είναι ο βασικός εκφραστικός τρόπος, με τ...",Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,[],[],Ο κυριότερος εκφραστικός τρόπος με τον οποίο ο...,arxaia,2025,GEL,open_ended,NaN,passage,10.0,,
